<a href="https://colab.research.google.com/github/mandara-samanalee/FYP_Team_Intellicore/blob/main/notebooks/module3_activity/V2_feature_extraction_%26_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Review Anomaly Detection Pipeline (v2 with Burstiness)

This notebook is designed to prepare data, extract advanced temporal features, and train an anomaly detection model targeting synthetic review campaigns.

### 🔑 Core Objectives
1. **Feature Extraction**: Process raw product reviews into time-based sliding windows (30-day windows, 7-day slide) and calculate five behavioral features—including **`burst_span_5`** (the tightest timespan containing 5 reviews) alongside standard count and Inter-Arrival Time (IAT) stats.
2. **Model Training**: Train an **Isolation Forest** model to establish a behavioral baseline and identify anomalous review surges.

### 📥 Inputs
* **`df_train.csv`**: A CSV dataset containing historical, raw reviews. It must include:
  * `product_id`: Unique identifier for each product.
  * `timestamp`: Unix time (in seconds) or datetime representation of when the review was posted.

### 📤 Outputs
* **`df_features_train_v2.csv`**: The extracted feature dataset containing window-based statistics (`review_count`, `mean_iat`, `std_iat`, `burstiness`, `burst_span_5`) for each product's timeline.
* **`isolation_forest_module3_v2.pkl`**: The trained Isolation Forest model object.
* **`module3_feature_cols_v2.pkl`**: A serialized list of the feature column names used during training.
* **`module3_product_thresholds_v2.pkl`**: Anomaly thresholds computed per product (based on the 5th percentile of their training anomaly scores).
* **`module3_global_threshold_v2.pkl`**: A fallback global threshold computed from all training window scores.

*Note: The subsequent evaluation, absolute threshold sweeps, and campaign-level recall calculations in this notebook are experimental zones used to fine-tune flag thresholds and diagnostic metrics.*
```

In [ ]:
# ============================================================
# MODULE 3 v2 — Feature extraction WITH burst feature
# Saves to experiment folder; originals untouched
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np, os

BASE = "/content/drive/MyDrive"
EXP  = "/content/drive/MyDrive/module3_burst_experiment"
os.makedirs(EXP, exist_ok=True)

df_train = pd.read_csv(f"{BASE}/df_train.csv")
df_train["datetime"] = pd.to_datetime(df_train["datetime"])
print(f"df_train: {len(df_train):,} records, {df_train['product_id'].nunique():,} products")

WINDOW_SECS = 30 * 24 * 3600
STEP_SECS   = 7  * 24 * 3600
BURST_N     = 5   # tightest span containing 5 reviews

print("Setup ready. Window 30d / step 7d / burst N=5")

Mounted at /content/drive
df_train: 324,446 records, 13,196 products
Setup ready. Window 30d / step 7d / burst N=5


In [ ]:
# ============================================================
# Extract 5 features per product-window:
#   review_count, mean_iat, std_iat, burstiness  (original 4)
#   burst_span_5  (NEW — shortest span containing 5 reviews, seconds)
# ============================================================

def tightest_span_in_window(wt, N):
    """Shortest span (seconds) containing N reviews within this window.
       If fewer than N reviews, return a large sentinel (no burst possible)."""
    if len(wt) < N:
        return float(WINDOW_SECS)  # sentinel: 'no tight cluster' = full window width
    spans = wt[N-1:] - wt[:-(N-1)]
    return float(spans.min())

print(f"Extracting features from df_train...")
print(f"Total products: {df_train['product_id'].nunique():,}\n")

all_windows = []
products = df_train["product_id"].unique()
total = len(products)

for idx, product_id in enumerate(products):
    if idx % 1000 == 0:
        print(f"  Processing product {idx:,} of {total:,}...")

    product_df = df_train[df_train["product_id"] == product_id].sort_values("timestamp")
    timestamps = product_df["timestamp"].values
    if len(timestamps) == 0:
        continue

    t_start, t_end = timestamps[0], timestamps[-1]
    window_start = t_start

    while window_start <= t_end:
        window_end = window_start + WINDOW_SECS
        mask = (timestamps >= window_start) & (timestamps < window_end)
        wt = timestamps[mask]
        count = len(wt)

        if count >= 2:
            iat = np.diff(wt)
            mean_iat = float(np.mean(iat))
            std_iat  = float(np.std(iat))
            burstiness = (std_iat - mean_iat) / (std_iat + mean_iat) if (std_iat + mean_iat) > 0 else 0.0
        else:
            mean_iat = std_iat = burstiness = 0.0

        burst_span_5 = tightest_span_in_window(wt, BURST_N)   # NEW feature

        all_windows.append({
            "product_id":   product_id,
            "window_start": pd.Timestamp(window_start, unit="s"),
            "window_end":   pd.Timestamp(window_end, unit="s"),
            "review_count": count,
            "mean_iat":     mean_iat,
            "std_iat":      std_iat,
            "burstiness":   burstiness,
            "burst_span_5": burst_span_5,        # NEW
            "is_generated": 0
        })
        window_start += STEP_SECS

df_features_train_v2 = pd.DataFrame(all_windows)

print(f"\nExtraction complete.")
print(f"Total windows: {len(df_features_train_v2):,}")
print(f"Columns: {list(df_features_train_v2.columns)}")
print(f"\nburst_span_5 stats (seconds):")
print(df_features_train_v2['burst_span_5'].describe())

# Save to experiment folder
df_features_train_v2.to_csv(f"{EXP}/df_features_train_v2.csv", index=False)
print(f"\nSaved to {EXP}/df_features_train_v2.csv")

Extracting features from df_train...
Total products: 13,196

  Processing product 0 of 13,196...
  Processing product 1,000 of 13,196...
  Processing product 2,000 of 13,196...
  Processing product 3,000 of 13,196...
  Processing product 4,000 of 13,196...
  Processing product 5,000 of 13,196...
  Processing product 6,000 of 13,196...
  Processing product 7,000 of 13,196...
  Processing product 8,000 of 13,196...
  Processing product 9,000 of 13,196...
  Processing product 10,000 of 13,196...
  Processing product 11,000 of 13,196...
  Processing product 12,000 of 13,196...
  Processing product 13,000 of 13,196...

Extraction complete.
Total windows: 893,238
Columns: ['product_id', 'window_start', 'window_end', 'review_count', 'mean_iat', 'std_iat', 'burstiness', 'burst_span_5', 'is_generated']

burst_span_5 stats (seconds):
count    8.932380e+05
mean     2.505874e+06
std      3.882239e+05
min      5.260000e+02
25%      2.592000e+06
50%      2.592000e+06
75%      2.592000e+06
max      2

In [ ]:
# ============================================================
# MODULE 3 v2 — Retrain Isolation Forest on 5 features
# Saves to experiment folder with _v2 names
# ============================================================
from sklearn.ensemble import IsolationForest
import joblib

# 5 features now (original 4 + burst_span_5)
non_feature_cols = ["product_id", "window_start", "window_end", "is_generated"]
feature_cols_v2 = [c for c in df_features_train_v2.columns if c not in non_feature_cols]

print("Feature columns (v2):", feature_cols_v2)

X_train = df_features_train_v2[feature_cols_v2].copy()
if X_train.isna().sum().sum() > 0:
    X_train = X_train.fillna(0)
    print("NaNs filled")
else:
    print("No NaNs")

# Same hyperparameters as your original model
iso_forest_v2 = IsolationForest(
    n_estimators=200,
    max_samples="auto",
    contamination=0.05,
    max_features=1.0,
    random_state=42,
    n_jobs=-1
)
print("\nTraining v2...")
iso_forest_v2.fit(X_train)
print("Done.")

# Anomaly scores + per-product thresholds (same logic as original)
df_features_train_v2["anomaly_score"] = iso_forest_v2.decision_function(X_train)

MIN_WINDOWS = 30
PERCENTILE  = 0.05
global_threshold_v2 = df_features_train_v2["anomaly_score"].quantile(PERCENTILE)

product_thresholds_v2 = {}
for pid, grp in df_features_train_v2.groupby("product_id"):
    if len(grp) >= MIN_WINDOWS:
        product_thresholds_v2[pid] = grp["anomaly_score"].quantile(PERCENTILE)
    else:
        product_thresholds_v2[pid] = global_threshold_v2

# Save all to experiment folder
joblib.dump(iso_forest_v2,        f"{EXP}/isolation_forest_module3_v2.pkl")
joblib.dump(feature_cols_v2,      f"{EXP}/module3_feature_cols_v2.pkl")
joblib.dump(product_thresholds_v2,f"{EXP}/module3_product_thresholds_v2.pkl")
joblib.dump(global_threshold_v2,  f"{EXP}/module3_global_threshold_v2.pkl")

print(f"\nGlobal threshold v2: {global_threshold_v2:.6f}")
print(f"Saved 4 files to {EXP}")

# Quick score separation preview
scores = df_features_train_v2["anomaly_score"]
print(f"\nTrain score distribution: median {scores.median():.4f}, "
      f"5th pct {scores.quantile(0.05):.4f}, min {scores.min():.4f}")

Feature columns (v2): ['review_count', 'mean_iat', 'std_iat', 'burstiness', 'burst_span_5']
No NaNs

Training v2...
Done.

Global threshold v2: 0.000000
Saved 4 files to /content/drive/MyDrive/module3_burst_experiment

Train score distribution: median 0.2345, 5th pct 0.0000, min -0.1369


In [ ]:
# ============================================================
# v2 EVALUATION — product-level precision/recall with burst feature
# ============================================================
import random as _r

# Load genuine + synthetic
df_val   = pd.read_csv(f"{BASE}/df_val.csv")
df_test  = pd.read_csv(f"{BASE}/df_test.csv")
df_synth = pd.read_csv(f"{BASE}/unified_synthetic_campaigns.csv")
gen_all  = pd.concat([df_train, df_val, df_test], ignore_index=True)
targeted_products = set(df_synth['product_id'].unique())

BURST_N = 5

def tightest_span_in_window(wt, N):
    if len(wt) < N: return float(WINDOW_SECS)
    spans = wt[N-1:] - wt[:-(N-1)]
    return float(spans.min())

def extract_5feat(timestamps, synth_flags=None):
    """Extract 5-feature windows over a timeline. If synth_flags given,
       also track whether each window contains synthetic reviews."""
    order = np.argsort(timestamps)
    ts = np.asarray(timestamps)[order]
    fl = np.asarray(synth_flags)[order] if synth_flags is not None else None
    if len(ts)==0: return []
    t_end=ts[-1]; ws=ts[0]; out=[]
    while ws<=t_end:
        we=ws+WINDOW_SECS
        mask=(ts>=ws)&(ts<we); wt=ts[mask]; c=len(wt)
        if c>=2:
            iat=np.diff(wt); m=float(iat.mean()); s=float(iat.std())
            b=(s-m)/(s+m) if (s+m)>0 else 0.0
        else: m=s=b=0.0
        row={'review_count':c,'mean_iat':m,'std_iat':s,'burstiness':b,
             'burst_span_5':tightest_span_in_window(wt,BURST_N)}
        if fl is not None:
            row['has_synth']=bool(fl[mask].any())
        out.append(row)
        ws+=STEP_SECS
    return out

def thr(pid):
    t = product_thresholds_v2.get(pid, global_threshold_v2)
    return t if t is not None else global_threshold_v2

# --- Recall: targeted products (merge genuine + synthetic timeline) ---
tp = 0; targeted_present = 0
for pid in targeted_products:
    g = gen_all[gen_all['product_id']==pid]['timestamp'].values
    s = df_synth[df_synth['product_id']==pid]['timestamp'].values
    if len(g)+len(s)==0: continue
    targeted_present += 1
    ts = np.concatenate([g,s])
    fl = np.concatenate([np.zeros(len(g)), np.ones(len(s))])
    wins = extract_5feat(ts, fl)
    wdf = pd.DataFrame(wins)
    sc = iso_forest_v2.decision_function(wdf[feature_cols_v2])
    # flag product if ANY window scores below its threshold
    if (sc < thr(pid)).sum() >= 1:
        tp += 1
recall = tp / targeted_present

# --- FP: genuine sample (no synthetic) ---
genuine_products = [p for p in gen_all['product_id'].unique()
                    if p not in targeted_products]
_r.seed(42)
gsample = _r.sample(genuine_products, min(2000, len(genuine_products)))
fp = 0; scored = 0
for pid in gsample:
    g = gen_all[gen_all['product_id']==pid]['timestamp'].values
    wins = extract_5feat(g)
    if not wins: continue
    scored += 1
    wdf = pd.DataFrame(wins)
    sc = iso_forest_v2.decision_function(wdf[feature_cols_v2])
    if (sc < thr(pid)).sum() >= 1:
        fp += 1
fp_rate = fp/scored
est_FP = fp_rate * len(genuine_products)
precision = tp/(tp+est_FP) if (tp+est_FP)>0 else 0
f1 = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0

print("="*50)
print("v2 RESULTS (with burst_span_5 feature)")
print("="*50)
print(f"Targeted products: {targeted_present}")
print(f"Caught (recall):   {tp}/{targeted_present} = {recall*100:.1f}%")
print(f"Genuine FP rate:   {fp}/{scored} = {fp_rate*100:.1f}%")
print(f"Precision (est):   {precision*100:.1f}%")
print(f"F1:                {f1*100:.1f}%")
print()
print("vs OLD model: ~99% recall but ~5% precision, ~96% FP")

v2 RESULTS (with burst_span_5 feature)
Targeted products: 635
Caught (recall):   635/635 = 100.0%
Genuine FP rate:   1861/2000 = 93.0%
Precision (est):   5.0%
F1:                9.5%

vs OLD model: ~99% recall but ~5% precision, ~96% FP


In [ ]:
# Why are genuine products still flagged? Check what's tripping them.
# Look at a few genuine products that got falsely flagged.

_r.seed(42)
gsample_check = _r.sample(genuine_products, 500)

genuine_scores_all = []
for pid in gsample_check:
    g = gen_all[gen_all['product_id']==pid]['timestamp'].values
    wins = extract_5feat(g)
    if not wins: continue
    wdf = pd.DataFrame(wins)
    sc = iso_forest_v2.decision_function(wdf[feature_cols_v2])
    t = thr(pid)
    flagged_windows = wdf[sc < t]
    for _, w in flagged_windows.iterrows():
        genuine_scores_all.append({
            'burst_span_5': w['burst_span_5'],
            'review_count': w['review_count'],
        })

gs = pd.DataFrame(genuine_scores_all)
print(f"Genuine windows that got FLAGGED: {len(gs):,}")
print(f"\nOf these flagged genuine windows, burst_span_5 distribution:")
print(gs['burst_span_5'].describe())
print(f"\nHow many flagged genuine windows have burst_span_5 = sentinel (2592000, i.e. NO real burst): "
      f"{(gs['burst_span_5'] >= 2591999).sum():,} / {len(gs):,}")
print(f"How many have a REAL tight cluster (burst_span_5 < 86400 = under 1 day): "
      f"{(gs['burst_span_5'] < 86400).sum():,}")
print(f"\nReview count of flagged genuine windows:")
print(gs['review_count'].describe())

Genuine windows that got FLAGGED: 2,400

Of these flagged genuine windows, burst_span_5 distribution:
count    2.400000e+03
mean     2.199208e+06
std      7.263807e+05
min      1.480200e+04
25%      2.166374e+06
50%      2.592000e+06
75%      2.592000e+06
max      2.592000e+06
Name: burst_span_5, dtype: float64

How many flagged genuine windows have burst_span_5 = sentinel (2592000, i.e. NO real burst): 1,758 / 2,400
How many have a REAL tight cluster (burst_span_5 < 86400 = under 1 day): 11

Review count of flagged genuine windows:
count    2400.000000
mean        3.879583
std         3.263488
min         2.000000
25%         2.000000
50%         3.000000
75%         5.000000
max        56.000000
Name: review_count, dtype: float64


In [ ]:
# THE FIX: the burst feature creates real score separation.
# Test flagging on absolute score threshold instead of 5th-percentile-per-product.
# A genuine window with no burst scores HIGH; a real burst scores LOW.

# First: what do burst vs genuine windows actually score now?
# Recompute with labels to see the separation in v2 model.

burst_window_scores = []
genuine_window_scores = []

for pid in list(targeted_products)[:635]:
    g = gen_all[gen_all['product_id']==pid]['timestamp'].values
    s = df_synth[df_synth['product_id']==pid]['timestamp'].values
    if len(g)+len(s)==0: continue
    ts = np.concatenate([g,s]); fl = np.concatenate([np.zeros(len(g)), np.ones(len(s))])
    wins = extract_5feat(ts, fl)
    wdf = pd.DataFrame(wins)
    sc = iso_forest_v2.decision_function(wdf[feature_cols_v2])
    for i, w in wdf.iterrows():
        if w['has_synth']:
            burst_window_scores.append(sc[i])

_r.seed(42)
for pid in _r.sample(genuine_products, 1000):
    g = gen_all[gen_all['product_id']==pid]['timestamp'].values
    wins = extract_5feat(g)
    if not wins: continue
    wdf = pd.DataFrame(wins)
    sc = iso_forest_v2.decision_function(wdf[feature_cols_v2])
    genuine_window_scores.extend(sc.tolist())

bw = np.array(burst_window_scores)
gw = np.array(genuine_window_scores)

print("BURST windows (contain synthetic):")
print(f"  median {np.median(bw):.4f}, 75th {np.percentile(bw,75):.4f}, max {bw.max():.4f}")
print("GENUINE windows:")
print(f"  median {np.median(gw):.4f}, 5th {np.percentile(gw,5):.4f}, min {gw.min():.4f}")
print()

# Now sweep an absolute threshold using product-level once-is-enough rule
def eval_abs_v2(thresh):
    tp=0; present=0
    for pid in targeted_products:
        g = gen_all[gen_all['product_id']==pid]['timestamp'].values
        s = df_synth[df_synth['product_id']==pid]['timestamp'].values
        if len(g)+len(s)==0: continue
        present+=1
        ts=np.concatenate([g,s])
        wins=extract_5feat(ts)
        sc=iso_forest_v2.decision_function(pd.DataFrame(wins)[feature_cols_v2])
        if (sc<thresh).sum()>=1: tp+=1
    recall=tp/present
    fp=0;scored=0
    _r.seed(42)
    for pid in _r.sample(genuine_products, 2000):
        g=gen_all[gen_all['product_id']==pid]['timestamp'].values
        wins=extract_5feat(g)
        if not wins: continue
        scored+=1
        sc=iso_forest_v2.decision_function(pd.DataFrame(wins)[feature_cols_v2])
        if (sc<thresh).sum()>=1: fp+=1
    fpr=fp/scored; estFP=fpr*len(genuine_products)
    prec=tp/(tp+estFP) if (tp+estFP)>0 else 0
    f1=2*prec*recall/(prec+recall) if (prec+recall)>0 else 0
    return recall,fpr,prec,f1

print("Abs threshold │ Recall │ Gen FP │ Precision │  F1")
print("──────────────┼────────┼────────┼───────────┼──────")
for t in [-0.10, -0.08, -0.06, -0.04, -0.02]:
    r,fp,p,f1 = eval_abs_v2(t)
    print(f"   {t:+.2f}      │ {r*100:5.1f}% │ {fp*100:5.1f}% │  {p*100:5.1f}%  │ {f1*100:4.1f}%")

BURST windows (contain synthetic):
  median -0.0451, 75th -0.0303, max 0.2352
GENUINE windows:
  median 0.2345, 5th 0.0094, min -0.1369

Abs threshold │ Recall │ Gen FP │ Precision │  F1
──────────────┼────────┼────────┼───────────┼──────
   -0.10      │  23.0% │   1.2% │   47.3%  │ 30.9%
   -0.08      │  40.8% │   3.1% │   39.1%  │ 39.9%
   -0.06      │  49.8% │   9.0% │   21.2%  │ 29.8%
   -0.04      │  69.8% │  23.6% │   12.6%  │ 21.3%
   -0.02      │  99.2% │  36.8% │   11.6%  │ 20.8%


In [ ]:
# Campaign-level recall on v2 at a high-precision threshold (-0.08)
# A campaign is caught if ANY of its windows fires.
THRESH = -0.08

prod_to_campaign = df_synth.groupby('product_id')['campaign_id'].agg(
    lambda s: s.mode().iloc[0]).to_dict()
prod_to_type = df_synth.groupby('product_id')['campaign_type'].agg(
    lambda s: s.mode().iloc[0]).to_dict()

caught_campaigns = set()
all_campaigns = set(df_synth['campaign_id'])

for pid in targeted_products:
    g = gen_all[gen_all['product_id']==pid]['timestamp'].values
    s = df_synth[df_synth['product_id']==pid]['timestamp'].values
    if len(g)+len(s)==0: continue
    ts=np.concatenate([g,s])
    wins=extract_5feat(ts)
    sc=iso_forest_v2.decision_function(pd.DataFrame(wins)[feature_cols_v2])
    if (sc < THRESH).sum() >= 1:
        caught_campaigns.add(prod_to_campaign[pid])

# by type
from collections import defaultdict
type_total=defaultdict(set); type_caught=defaultdict(set)
for cid in all_campaigns:
    ct = df_synth[df_synth['campaign_id']==cid]['campaign_type'].iloc[0]
    type_total[ct].add(cid)
    if cid in caught_campaigns:
        type_caught[ct].add(cid)

print(f"CAMPAIGN-LEVEL recall at threshold {THRESH} (precision ~39% at window level):")
print(f"  Overall: {len(caught_campaigns)}/{len(all_campaigns)} = "
      f"{len(caught_campaigns)/len(all_campaigns)*100:.1f}%")
for ct in sorted(type_total):
    print(f"  Type {ct}: {len(type_caught[ct])}/{len(type_total[ct])} "
          f"({len(type_caught[ct])/len(type_total[ct])*100:.0f}%)")

CAMPAIGN-LEVEL recall at threshold -0.08 (precision ~39% at window level):
  Overall: 105/115 = 91.3%
  Type A: 31/31 (100%)
  Type B: 28/32 (88%)
  Type C: 19/20 (95%)
  Type D: 27/32 (84%)
